# GPS Attendance API
### Kiểm tải trên Kaggle CPU · 100 / 500 / 1.000 sinh viên

Đo hiệu năng API điểm danh bằng **Locust**, với **PostgreSQL + FastAPI (1 worker)** chạy trực tiếp trên Kaggle.

---

## Chuẩn bị và chạy

1. Push mã backend lên GitHub.
2. Upload notebook vào Kaggle.
3. Chọn **Accelerator: None**, bật **Internet**.
4. Chọn **Run All** hoặc chạy lần lượt các cell bên dưới.

**Không cần Docker.**

## Kịch bản kiểm tải

| Thành phần | Cấu hình |
| :--- | :--- |
| **Các mức tải** | 100 → 500 → 1.000 sinh viên |
| **Dữ liệu nền** | Một lớp có 1.000 tài khoản sinh viên riêng biệt |
| **Cách gửi tải** | Chờ đủ người dùng giả lập rồi gửi một đợt điểm danh |
| **Số request** | Mỗi sinh viên gửi đúng một request, không tự retry |
| **Phiên và xác thực** | Phiên mới cho mỗi mức tải; token riêng cho từng sinh viên |
| **Giới hạn chờ** | 90 giây mỗi mức; kết thúc sớm khi request hoàn tất |
| **Tọa độ** | Dữ liệu mẫu nằm trong vùng điểm danh |


## Kết quả đầu ra

- **Requests / Failures:** số request và số lỗi được ghi nhận.
- **p95 (ms):** thời gian phản hồi mà 95% request không vượt quá.
- **HTML và CSV:** báo cáo riêng cho từng mức tải.
- **ZIP:** toàn bộ báo cáo và cấu hình đo để tải về.

---



1. Upload backend, cài PostgreSQL và tạo môi trường.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/gps-loadtest")
REPO = WORK / "repo"
VENV = WORK / "venv"
WORK.mkdir(parents=True, exist_ok=True)

def run(*args, **kwargs):
    return subprocess.run(
        [str(a) for a in args], check=True, **kwargs
    )

if not REPO.exists():
    run(
        "git", "clone",
        "https://github.com/hhkc861/gps-attendance-system.git",
        REPO,
    )

run("apt-get", "update", "-qq")
run(
    "apt-get", "install", "-y", "-qq",
    "postgresql",
    f"python{sys.version_info.major}.{sys.version_info.minor}-venv",
)

run(sys.executable, "-m", "venv", VENV)
PYTHON = VENV / "bin/python"

run(
    PYTHON, "-m", "pip", "install", "-q",
    "-r", REPO / "requirements.txt",
    "psycopg[binary]", "locust==2.32.9", "wrapt",
)

COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()

print("Backend commit:", COMMIT)


2. Khởi động PostgreSQL, cấu hình kết nối và tạo các bảng của backend.


In [ ]:
import secrets

DB_PASSWORD = globals().get("DB_PASSWORD") or secrets.token_hex(16)
JWT_SECRET = globals().get("JWT_SECRET") or secrets.token_hex(32)

run("service", "postgresql", "start")

def psql(sql):
    return subprocess.check_output(
        [
            "runuser", "-u", "postgres", "--",
            "psql", "-v", "ON_ERROR_STOP=1", "-tAc", sql,
        ],
        text=True,
    ).strip()

if not psql("SELECT 1 FROM pg_roles WHERE rolname='gps_loadtest'"):
    psql("CREATE ROLE gps_loadtest LOGIN")

psql(f"ALTER ROLE gps_loadtest PASSWORD '{DB_PASSWORD}'")

if not psql("SELECT 1 FROM pg_database WHERE datname='gps_loadtest'"):
    psql("CREATE DATABASE gps_loadtest OWNER gps_loadtest")

ENV = os.environ.copy()
ENV.update({
    "DATABASE_URL": (
        f"postgresql+psycopg://gps_loadtest:{DB_PASSWORD}"
        "@127.0.0.1:5432/gps_loadtest"
    ),
    "JWT_SECRET": JWT_SECRET,
    "PYTHONPATH": str(REPO),
})
ENV.pop("BOOTSTRAP_ADMIN_PASSWORD", None)

run(
    PYTHON, "-c",
    "from database import connect_database, close_database; "
    "connect_database(); close_database()",
    cwd=REPO, env=ENV,
)

print("PostgreSQL và các bảng đã sẵn sàng.")


3. Chuẩn bị một lớp với 1.000 tài khoản sinh viên.


In [ ]:
SEED = WORK / "seed.py"

SEED.write_text(r'''
import json
from pathlib import Path
from sqlalchemy import select
from database import SessionLocal
from models import User, Section

from utils.security import hash_password

names = [f"kaggle_student_{i:04d}" for i in range(1000)]

with SessionLocal() as db:
    users = {
        u.username: u
        for u in db.scalars(
            select(User).where(User.username.in_(names))
        ).all()
    }

    missing = [name for name in names if name not in users]
    if missing:
        password_hash = hash_password("KaggleLoadTest123!")
        for name in missing:
            user = User(
                username=name,
                full_name=name,
                password_hash=password_hash,
                role="STUDENT",
            )
            db.add(user)
            users[name] = user
        db.flush()

    section = db.scalar(
        select(Section).where(
            Section.section_code == "KAGGLE_LOADTEST_1000"
        )
    )
    if section is None:
        section = Section(section_code="KAGGLE_LOADTEST_1000")
        db.add(section)

    section.students = [users[name] for name in names]
    db.commit()

    data = {
        "section_id": section.id,
        "lat": 21.03824,
        "lng": 105.78256,
        "accounts": [
            {"student_id": users[name].id}
            for name in names
        ],
    }
    Path("/kaggle/working/gps-loadtest/accounts.json").write_text(
        json.dumps(data)
    )

print("Đã tạo lớp kiểm tải với 1.000 sinh viên.")
''', encoding="utf-8")

run(PYTHON, SEED, cwd=REPO, env=ENV)


4. Chạy FastAPI bằng Uvicorn với 1 worker tại cổng 8000.


In [ ]:
import time

if "api" in globals() and api.poll() is None:
    api.terminate()
    api.wait(timeout=30)

with (WORK / "api.log").open("w") as log:
    api = subprocess.Popen(
        [
            str(PYTHON), "-m", "uvicorn", "main:app",
            "--host", "127.0.0.1", "--port", "8000",
            "--workers", "1",
        ],
        cwd=REPO,
        env=ENV,
        stdout=log,
        stderr=subprocess.STDOUT,
    )

time.sleep(5)
print("Đã khởi chạy API tại http://127.0.0.1:8000")


5. Mỗi sinh viên điểm danh một lần; kiểm tra HTTP và nội dung kết quả.


In [ ]:
LOCUST = WORK / "locustfile.py"

LOCUST.write_text(r'''
import json
import os
from pathlib import Path

import gevent
from gevent.event import Event
from locust import HttpUser, task

data = json.loads(Path(os.environ["LOADTEST_INPUT"]).read_text())
accounts = iter(data["accounts"])
target = len(data["accounts"])
ready = completed = 0
gate = Event()
park = Event()


class Student(HttpUser):
    def on_start(self):
        global ready
        account = next(accounts)
        self.student_id = account["student_id"]
        self.client.headers["Authorization"] = "Bearer " + account["token"]
        ready += 1
        if ready == target:
            gate.set()

    @task
    def check_in_once(self):
        global completed
        gate.wait()

        with self.client.post(
            f'/api/sessions/{data["session_id"]}/check-in',
            json={"lat": data["lat"], "lng": data["lng"]},
            name="/api/sessions/[id]/check-in",
            timeout=30,
            catch_response=True,
        ) as response:
            if response.status_code == 200:
                try:
                    body = response.json()
                    if not (
                        body["status"] == "PRESENT"
                        and body["studentId"] == self.student_id
                        and body["sessionId"] == data["session_id"]
                        and isinstance(body["recordId"], int)
                    ):
                        response.failure("Sai kết quả điểm danh")
                except (ValueError, KeyError, TypeError):
                    response.failure("JSON điểm danh không hợp lệ")

        completed += 1
        if completed == target:
            gevent.spawn_later(0.2, self.environment.runner.quit)

        park.wait()
''', encoding="utf-8")

print("Đã tạo kịch bản Locust.")


6. Chạy lần lượt 100, 500 và 1.000 sinh viên, tạo lại phiên cho mỗi mức.


In [ ]:
import csv
import json

RESULTS = WORK / f"results_{time.time_ns()}"
RESULTS.mkdir()
INPUT = WORK / "run_input.json"

PREPARE = r'''
import json
import sys
from pathlib import Path
from datetime import timedelta
from database import SessionLocal
from models import AttendanceSession, now_utc
from utils.security import create_access_token

work = Path("/kaggle/working/gps-loadtest")
data = json.loads((work / "accounts.json").read_text())
n = int(sys.argv[1])

with SessionLocal() as db:
    now = now_utc()
    session = AttendanceSession(
        section_id=data["section_id"],
        center_lat=data["lat"],
        center_lng=data["lng"],
        radius_meters=100,
        start_time=now - timedelta(minutes=1),
        end_time=now + timedelta(hours=1),
    )
    db.add(session)
    db.commit()
    data["session_id"] = session.id

data["accounts"] = [
    {
        "student_id": a["student_id"],
        "token": create_access_token(sub=str(a["student_id"])),
    }
    for a in data["accounts"][:n]
]
target = work / "run_input.json"
target.write_text(json.dumps(data))
target.chmod(0o600)
'''

summary = []

for n in (100, 500, 1000):
    print(f"\n=== {n} sinh viên ===", flush=True)

    run(PYTHON, "-c", PREPARE, n, cwd=REPO, env=ENV)

    result = subprocess.run(
        [
            str(PYTHON), "-m", "locust",
            "-f", str(LOCUST),
            "--headless",
            "--host", "http://127.0.0.1:8000",
            "-u", str(n), "-r", "1000",
            "--run-time", "90s",
            "--stop-timeout", "35",
            "--only-summary",
            "--csv", str(RESULTS / str(n)),
            "--html", str(RESULTS / f"{n}.html"),
        ],
        cwd=REPO,
        env={**ENV, "LOADTEST_INPUT": str(INPUT)},
        timeout=150,
    )

    with (RESULTS / f"{n}_stats.csv").open(newline="") as file:
        stats = next(
            row for row in csv.DictReader(file)
            if row["Name"] == "Aggregated"
        )

    requests = int(stats["Request Count"])
    failures = int(stats["Failure Count"])
    row = {
        "students": n,
        "requests": requests,
        "failures": failures,
        "p95_ms": stats["95%"],
        "result": "OK" if (
            requests == n
            and failures == 0
            and result.returncode == 0
        ) else "CHECK",
    }
    summary.append(row)

    with (RESULTS / "summary.csv").open("w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=row.keys())
        writer.writeheader()
        writer.writerows(summary)

    INPUT.unlink(missing_ok=True)
    print(
        f'{row["result"]}: {requests}/{n} requests, '
        f'{failures} lỗi, p95={row["p95_ms"]} ms',
        flush=True,
    )

    if n != 1000:
        time.sleep(5)

print("\nĐã lưu kết quả tại:", RESULTS)


7. Hiển thị kết quả, tải báo cáo HTML/CSV/ZIP và dừng API.


In [ ]:
import shutil
from IPython.display import FileLink, display

for row in summary:
    print(
        f'{row["students"]} sinh viên: '
        f'{row["requests"]} requests, '
        f'{row["failures"]} lỗi, '
        f'p95={row["p95_ms"]} ms — {row["result"]}'
    )

(RESULTS / "config.json").write_text(
    json.dumps({
        "backend_commit": COMMIT,
        "workers": 1,
        "class_size": 1000,
        "levels": [100, 500, 1000],
        "scenario": "Mỗi sinh viên điểm danh một lần, phiên mới mỗi mức",
        "environment": "API, PostgreSQL và Locust cùng máy Kaggle",
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

archive = shutil.make_archive(
    str(RESULTS.parent / f"{RESULTS.name}_reports"),
    "zip",
    root_dir=RESULTS,
)

print("\nTải toàn bộ báo cáo:")
display(FileLink(archive))

print("\nBáo cáo HTML từng mức:")
for n in (100, 500, 1000):
    display(FileLink(str(RESULTS / f"{n}.html")))

if api.poll() is None:
    api.terminate()
    api.wait(timeout=30)

print("\nĐã dừng API.")